In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install kagglehub
!pip install pandas numpy tensorflow scikit-learn

In [4]:
import kagglehub

path = kagglehub.dataset_download(
    "grouplens/movielens-20m-dataset"
)

print(path)

100%|██████████| 195M/195M [00:01<00:00, 195MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/grouplens/movielens-20m-dataset/versions/1


In [5]:
import shutil
import os

drive_dataset = "/content/drive/MyDrive/MovieLens20M"

if not os.path.exists(drive_dataset):
    shutil.copytree(
        path,
        drive_dataset,
        dirs_exist_ok=True
    )

print("Dataset Saved")

Dataset Saved


In [6]:
import pandas as pd

dataset_path = "/content/drive/MyDrive/MovieLens20M"

ratings = pd.read_csv(
    dataset_path + "/rating.csv"
)

movies = pd.read_csv(
    dataset_path + "/movie.csv"
)

print(ratings.head())

   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40


In [7]:
ratings = ratings.sample(
    2000000,
    random_state=42
)

In [8]:
from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

ratings["user"] = user_encoder.fit_transform(
    ratings["userId"]
)

ratings["movie"] = movie_encoder.fit_transform(
    ratings["movieId"]
)

In [9]:
X = ratings[["user","movie"]]

y = ratings["rating"]

In [10]:
import tensorflow as tf

num_users = ratings["user"].nunique()
num_movies = ratings["movie"].nunique()

user_input = tf.keras.layers.Input(shape=(1,))
movie_input = tf.keras.layers.Input(shape=(1,))

user_embed = tf.keras.layers.Embedding(
    num_users,
    50
)(user_input)

movie_embed = tf.keras.layers.Embedding(
    num_movies,
    50
)(movie_input)

x = tf.keras.layers.Concatenate()(
    [
        tf.keras.layers.Flatten()(user_embed),
        tf.keras.layers.Flatten()(movie_embed)
    ]
)

x = tf.keras.layers.Dense(
    128,
    activation="relu"
)(x)

output = tf.keras.layers.Dense(1)(x)

model = tf.keras.Model(
    [user_input,movie_input],
    output
)

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 50)     │  6,784,850 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 50)     │    885,950 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 100)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     12,928 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        129 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,683,857 (29.31 MB)

 Trainable params: 7,683,857 (29.31 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
import os

checkpoint_dir = "/content/drive/MyDrive/movie_checkpoints"

os.makedirs(
    checkpoint_dir,
    exist_ok=True
)

In [12]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath=checkpoint_dir + "/epoch_{epoch:02d}.keras",
    save_freq="epoch"
)

In [13]:
from tensorflow.keras.callbacks import CSVLogger

csv_logger = CSVLogger(
    checkpoint_dir + "/training.csv"
)

In [14]:
history = model.fit(
    [
        X["user"],
        X["movie"]
    ],
    y,
    epochs=20,
    batch_size=1024,
    callbacks=[
        checkpoint,
        csv_logger
    ]
)

Epoch 1/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 1.1376
Epoch 2/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.7337
Epoch 3/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.6862
Epoch 4/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.6503
Epoch 5/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.6084
Epoch 6/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.5629
Epoch 7/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.5142
Epoch 8/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.4600
Epoch 9/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.4047
Epoch 10/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.3545
Epoch 11/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.3115
Epoch 12/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.2761
Epoch 13/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.2470
Epoch 14/20
1954/1954 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.2228
Epoch 15/20
1954/1954 ━━━━━━

In [15]:
model.save(
"/content/drive/MyDrive/final_movie_model.keras"
)

In [16]:
import os

os.path.exists(
"/content/drive/MyDrive/final_movie_model.keras"
)

True

In [17]:
import pickle

with open(
    "/content/drive/MyDrive/user_encoder.pkl",
    "wb"
) as f:
    pickle.dump(user_encoder, f)

with open(
    "/content/drive/MyDrive/movie_encoder.pkl",
    "wb"
) as f:
    pickle.dump(movie_encoder, f)

In [18]:
movie_lookup = movies.set_index(
    "movieId"
)["title"].to_dict()

In [19]:
import numpy as np

def recommend_movies(user_id, top_n=10):

    encoded_user = user_encoder.transform(
        [user_id]
    )[0]

    all_movies = ratings["movie"].unique()

    user_array = np.full(
        len(all_movies),
        encoded_user
    )

    predictions = model.predict(
        [user_array, all_movies],
        verbose=0
    )

    top_indices = predictions.flatten().argsort()[-top_n:]

    recommended = all_movies[top_indices]

    original_movie_ids = movie_encoder.inverse_transform(
        recommended
    )

    return [
        movie_lookup.get(mid)
        for mid in original_movie_ids
    ]

In [20]:
recommend_movies(
    user_id=1,
    top_n=10
)

['White Squall (1996)',
 'Betrayed (1988)',
 'Life Is Rosy (a.k.a. Life Is Beautiful) (Vie est belle, La) (1987)',
 'Dilwale Dulhania Le Jayenge (1995)',
 '5 Deadly Venoms, The (Wu du) (Five Deadly Venoms) (1978)',
 'Beautiful Thing (1996)',
 'Savannah Smiles (1982)',
 'Cirque du Soleil: Worlds Away (2012)',
 'Gardens of Stone (1987)',
 'Heidi (1937)']

In [29]:
import numpy as np
from sklearn.metrics import mean_squared_error

user_test = np.array(
    X["user"]
)

movie_test = np.array(
    X["movie"]
)

pred = model.predict(
    [user_test, movie_test],
    verbose=0
)

rmse = np.sqrt(
    mean_squared_error(
        y,
        pred
    )
)

print("RMSE:", rmse)

RMSE: 0.3317345145544727


In [23]:
print(model.input_shape)
print(model.inputs)

[(None, 1), (None, 1)]
[<KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor>, <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1>]


In [25]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

sample_size = 1000

user_sample = X["user"].values[:sample_size].astype("int32")
movie_sample = X["movie"].values[:sample_size].astype("int32")

pred = model.predict(
    [user_sample, movie_sample],
    verbose=0
)

true_values = y[:sample_size]

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        pred
    )
)

mae = mean_absolute_error(
    true_values,
    pred
)

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 0.3106165264183132
MAE: 0.23317390543222427


In [26]:
from sklearn.model_selection import train_test_split

X_train_user, X_test_user, X_train_movie, X_test_movie, y_train, y_test = train_test_split(
    X["user"].values,
    X["movie"].values,
    y.values,
    test_size=0.2,
    random_state=42
)

In [27]:
pred = model.predict(
    [
        X_test_user.astype("int32"),
        X_test_movie.astype("int32")
    ],
    verbose=0
)

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred
    )
)

mae = mean_absolute_error(
    y_test,
    pred
)

print("Test RMSE:", rmse)
print("Test MAE:", mae)

Test RMSE: 0.3317167760449161
Test MAE: 0.24112748871073128


In [32]:
import tensorflow as tf

model.fit(
    [
        tf.convert_to_tensor(X_train_user.reshape(-1, 1), dtype=tf.int32),
        tf.convert_to_tensor(X_train_movie.reshape(-1, 1), dtype=tf.int32)
    ],
    tf.convert_to_tensor(y_train.reshape(-1, 1), dtype=tf.float32),
    epochs=20,
    batch_size=1024
)

Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.1281
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1151
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1070
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.1006
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0950
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.0899
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.0857
Epoch 8/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0818
Epoch 9/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0781
Epoch 10/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0751
Epoch 11/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0722
Epoch 12/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0695
Epoch 13/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0672
Epoch 14/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0649
Epoch 15/20
1563/1563 ━━━━

In [33]:
import numpy as np

def recommend_movies(user_id, top_n=10):

    encoded_user = user_encoder.transform(
        [user_id]
    )[0]

    movie_ids = ratings["movie"].unique()

    users = np.full(
        len(movie_ids),
        encoded_user
    )

    scores = model.predict(
        [users, movie_ids],
        verbose=0
    )

    top_movies = movie_ids[
        scores.flatten().argsort()[-top_n:]
    ]

    original_ids = movie_encoder.inverse_transform(
        top_movies
    )

    return movies[
        movies["movieId"].isin(original_ids)
    ][["title"]]

In [34]:
recommend_movies(
    user_id=1,
    top_n=10
)

,title
57,"Postman, The (Postino, Il) (1994)"
4017,Gardens of Stone (1987)
5003,Heidi (1937)
7382,"Fountainhead, The (1949)"
7726,"Crazy Stranger, The (Gadjo Dilo) (1997)"
8715,Johnny Got His Gun (1971)
10768,Project A ('A' gai waak) (1983)
12025,12:08 East of Bucharest (A fost sau n-a fost?)...
19215,Beasts of the Southern Wild (2012)
19875,Indie Game: The Movie (2012)


In [35]:
recommended = recommend_movies(1)

recommended.to_csv(
"/content/drive/MyDrive/recommendations.csv",
index=False
)